In [1]:
import logging
import os
import sys
from io import BytesIO
import pandas as pd
import requests
from datetime import date, timedelta, datetime
from collections import Counter
import numpy as np
import json
from astroquery.vizier import Vizier
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from tqdm import tqdm
import time
from astropy.table import Table, Column, vstack, join
import re
from astropy.time import Time
from glob import glob

In [2]:
def load_fastspec_table(file_path):
    """Load the FASTSPEC table from a FITS file."""
    with fits.open(file_path) as hdul:
        return Table(hdul['FASTSPEC'].data)

def load_metadata_table(file_path):
    """Load the FASTSPEC table from a FITS file."""
    with fits.open(file_path) as hdul:
        return Table(hdul['METADATA'].data)

def load_specphot_table(file_path):
    """Load the FASTSPEC table from a FITS file."""
    with fits.open(file_path) as hdul:
        return Table(hdul['SPECPHOT'].data)

In [ ]:
loa_path = "/global/cfs/cdirs/desi/vac/dr1/fastspecfit/iron/v3.0/catalogs"
fastspec_files = np.sort(glob(os.path.join(loa_path, "*.fits")))
loa_metadata = vstack([load_metadata_table(os.path.join(loa_path, i)) for i in fastspec_files])

In [ ]:
loa_fastspec = vstack([load_fastspec_table(os.path.join(loa_path, i)) for i in fastspec_files])

In [ ]:
loa_fastphot = vstack([load_specphot_table(os.path.join(loa_path, i)) for i in fastspec_files])

In [ ]:
loa_positions = loa_metadata["TARGETID", "SURVEY", "PROGRAM", "HEALPIX", "TILEID_LIST", "RA", "DEC", "Z", "ZWARN", "DELTACHI2", "SPECTYPE", "SUBTYPE", "Z_RR", "ZWARN_RR", "LS_ID"]

In [ ]:
iron_temp

In [ ]:
cols = [
    "HALPHA",
    "HBETA",
    "OIII_5007",
    "OII_3726",
    "OI_6300",
    "NII_6584",
    "SII_6716",
]

values = ["AMP", "FLUX", "BOXFLUX", "SIGMA", "VSHIFT", "EW", "CONT"]

specific = ["FLUX_LIMIT", "CHI2"]

all_cols = []
for line in cols:
    all_cols.extend([f"{line}_{v}" for v in values])

ivar_cols = [f"{c}_IVAR" for c in all_cols]

specific_cols = []
for line in cols:
    specific_cols.extend([f"{line}_{v}" for v in specific])

loa_fastspec_cut = loa_fastspec[["TARGETID", "SURVEY", "PROGRAM", "HEALPIX", "SNR_B", "SNR_R", "SNR_Z"] + all_cols + ivar_cols]


In [ ]:
loa_fastphot_cut = loa_fastphot["TARGETID", "SURVEY", "PROGRAM", "HEALPIX", "VDISP", "VDISP_IVAR", "LOGMSTAR", "LOGMSTAR_IVAR"]

In [ ]:
iron_temp = join(
    loa_positions,
    loa_fastphot_cut,
    keys=["TARGETID", "SURVEY", "PROGRAM", "HEALPIX"],
    join_type="inner"
)

iron_all = join(
    iron_temp,
    loa_fastspec_cut,
    keys=["TARGETID", "SURVEY", "PROGRAM", "HEALPIX"],
    join_type="inner"
)

In [ ]:
import numpy as np

t = iron_all

# --- ZWARN preference: 1 if good, 0 if bad ---
zwarn_good = (t["ZWARN"] == 0).astype(int)

# --- compute min SNR across cameras ---
snr_stack = np.vstack([
    np.array(t["SNR_B"], dtype=float),
    np.array(t["SNR_R"], dtype=float),
    np.array(t["SNR_Z"], dtype=float),
])

snr_min = np.nanmin(snr_stack, axis=0)

# --- build final score ---
# ZWARN dominates; SNR breaks ties
score = zwarn_good * 1e6 + snr_min

t["_rank_score"] = score

In [75]:
t.sort(["TARGETID", "_rank_score"])
t = t[::-1]   # descending within TARGETID

In [76]:
_, first_idx = np.unique(np.array(t["TARGETID"]), return_index=True)
iron_all_1 = t[np.sort(first_idx)]

In [80]:
len(np.unique(loa_positions["TARGETID"]))

17362235

In [81]:
iron_all_1.remove_column("_rank_score")

In [83]:
iron_all_1.write("iron_all_1.fits", format="fits", overwrite=True)